In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the new dataset (Update the filename to match your file)
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# Standardise column spacing and strip whitespace to prevent key errors
df7.columns = df7.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# FIX: Clear out any pre-existing calculated column copies to prevent structural duplication errors
cols_to_clear = ['Margin_Percentage', 'Winner_Votes', 'Runner_Up_Votes', 'Margin_Of_Victory', 'Winner_Party', 'Cluster_ID']
df7 = df7.drop(columns=[c for c in cols_to_clear if c in df7.columns], errors='ignore')

# 2. Select the full array of primary political party columns from your new schema
core_parties = [
    'Dravida Munnetra Kazhagam',
    'All India Anna Dravida Munnetra Kazhagam', 
    'Tamilaga Vettri Kazhagam', 
    'Naam Tamilar Katchi', 
    'Bahujan Samaj Party', 
    'Puthiya Tamilagam',
    'National Maha Sabha Party', 
    'Thamizhaka Padaippalar Makkal Katchi',
    'All India Puratchi Thalaivar Makkal Munnettra Kazhagam', 
    'Samaniya Makkal Nala Katchi', 
    'Tamizhaga Vaazhvurimai Katchi',
    'All India Jananayaka Makkal Kazhagam', 
    'Vidial Valarchi Perani'
]

# Clean missing numerical fields by filling with 0
df7[core_parties] = df7[core_parties].fillna(0)

# Exact structural column definitions from your new dataset
station_col = 'Serial No. Of Polling Station'
building_col = 'Location and name of Building in which Polling Station located'
area_col = 'Polling Area'

# Explicitly isolate the 7 Independent columns present in your data (Independent to Independent.6)
independent_candidate_cols = ['Independent', 'Independent.1', 'Independent.2', 'Independent.3', 'Independent.4', 'Independent.5', 'Independent.6']
# Ensure independent columns that exist are treated as floats/ints and filled with 0 if blank
existing_ind_cols = [c for c in independent_candidate_cols if c in df7.columns]
df7[existing_ind_cols] = df7[existing_ind_cols].fillna(0)

# Calculate total independent votes dynamically across all 7 tracking lanes
df7['Total_Independent_Votes'] = df7[existing_ind_cols].sum(axis=1)

# 3. Calculate true total votes for normalization (13 Core Parties + 7 Independents + NOTA)
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['Total_Independent_Votes'].fillna(0) + df7['NOTA'].fillna(0)

# Filter out empty entries to completely avoid division by zero errors
df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Create normalized percentage shares (%)
share_cols = []
for party in core_parties:
    # Generates clean, unique short codes using uppercase initials to prevent name clashes
    party_label = "".join([word[0].upper() for word in party.split() if word.isalpha()])
    col_name = f'{party_label}_share_pct'
    
    if col_name in df7.columns:
        df7 = df7.drop(columns=[col_name])
        
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Append strategic independent voting dimensions
df7['independent_share_pct'] = (df7['Total_Independent_Votes'].fillna(0) / df7['Total_Calculated_Votes']) * 100

# Re-calculate fresh strategic voting margin metrics across the 13 active parties
df7['Winner_Votes'] = df7[core_parties].max(axis=1)
sorted_votes = np.sort(df7[core_parties].values, axis=1)
df7['Runner_Up_Votes'] = sorted_votes[:, -2]
df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']

# Isolate target metrics into a distinct tracking block to eliminate alignment assignment errors
X = df7[feature_cols].copy().fillna(0)

# 5. Extract and Scale features for the ML model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown to help map the text identities
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df7.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated for all 4 clusters.")


/Users/karthickkumarasamy/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (



--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            DMK_share_pct  AIADMK_share_pct  TVK_share_pct  NTK_share_pct  \
Cluster_ID                                                                  
0                   30.94             13.62           0.06           3.22   
1                   26.21             15.30           2.25           1.61   
2                   28.05             11.23           0.08           2.58   
3                   44.20              7.00           0.05           2.03   

            BSP_share_pct  PT_share_pct  NMSP_share_pct  TPMK_share_pct  \
Cluster_ID                                                                
0                    0.10          0.02            0.07            0.08   
1                    0.10          0.03            0.07            0.03   
2                    0.17          0.08            0.07            0.04   
3                    0.13          0.03            0.05            0.02   

            AIPTMMK_share

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Load the dataset
df7 = pd.read_csv("Detailed_Polling_Data.csv")

# Standardise column spacing and strip whitespace
df7.columns = df7.columns.str.replace(r'\s+', ' ', regex=True).str.strip()

# Clear out any pre-existing calculated column copies to prevent structural duplication errors
cols_to_clear = ['Margin_Percentage', 'Winner_Votes', 'Runner_Up_Votes', 'Margin_Of_Victory', 'Winner_Party', 'Cluster_ID']
df7 = df7.drop(columns=[c for c in cols_to_clear if c in df7.columns], errors='ignore')

# 2. Select the full array of primary political party columns from your new schema
core_parties = [
    'Dravida Munnetra Kazhagam',
    'All India Anna Dravida Munnetra Kazhagam', 
    'Tamilaga Vettri Kazhagam', 
    'Naam Tamilar Katchi', 
    'Bahujan Samaj Party', 
    'Puthiya Tamilagam',
    'National Maha Sabha Party', 
    'Thamizhaka Padaippalar Makkal Katchi',
    'All India Puratchi Thalaivar Makkal Munnettra Kazhagam', 
    'Samaniya Makkal Nala Katchi', 
    'Tamizhaga Vaazhvurimai Katchi',
    'All India Jananayaka Makkal Kazhagam', 
    'Vidial Valarchi Perani'
]

df7[core_parties] = df7[core_parties].fillna(0)

# Exact structural column definitions from your new dataset
station_col = 'Serial No. Of Polling Station'
building_col = 'Location and name of Building in which Polling Station located'
area_col = 'Polling Area'

# Explicitly isolate the 7 Independent columns present in your data
independent_candidate_cols = ['Independent', 'Independent.1', 'Independent.2', 'Independent.3', 'Independent.4', 'Independent.5', 'Independent.6']
existing_ind_cols = [c for c in independent_candidate_cols if c in df7.columns]
df7[existing_ind_cols] = df7[existing_ind_cols].fillna(0)

df7['Total_Independent_Votes'] = df7[existing_ind_cols].sum(axis=1)

# 3. Calculate true total votes for normalization (13 Core Parties + 7 Independents + NOTA)
df7['Total_Calculated_Votes'] = df7[core_parties].sum(axis=1) + df7['Total_Independent_Votes'].fillna(0) + df7['NOTA'].fillna(0)

df7 = df7[df7['Total_Calculated_Votes'] > 0].copy()

# 4. Feature Engineering: Create normalized percentage shares (%) with UNIQUE labels
share_cols = []
for party in core_parties:
    # Special naming override to handle TVK vs its alliance partner safely
    if party == 'Tamilaga Vettri Kazhagam':
        party_label = 'TVK'
    elif party == 'Tamizhaga Vaazhvurimai Katchi':
        party_label = 'TVK_ALLIANCE'
    else:
        party_label = "".join([word.upper() for word in party.split() if word.isalpha()])
        
    col_name = f'{party_label}_share_pct'
    
    if col_name in df7.columns:
        df7 = df7.drop(columns=[col_name])
        
    df7[col_name] = (df7[party] / df7['Total_Calculated_Votes']) * 100
    share_cols.append(col_name)

# Append strategic independent voting dimensions
df7['independent_share_pct'] = (df7['Total_Independent_Votes'].fillna(0) / df7['Total_Calculated_Votes']) * 100

# Re-calculate fresh strategic voting margin metrics across the 13 active parties
df7['Winner_Votes'] = df7[core_parties].max(axis=1)
sorted_votes = np.sort(df7[core_parties].values, axis=1)
df7['Runner_Up_Votes'] = sorted_votes[:, -2]
df7['Margin_Of_Victory'] = df7['Winner_Votes'] - df7['Runner_Up_Votes']
df7['Winner_Party'] = df7[core_parties].idxmax(axis=1)
df7['Margin_Percentage'] = (df7['Margin_Of_Victory'] / df7['Total_Calculated_Votes']) * 100

feature_cols = share_cols + ['independent_share_pct', 'Margin_Percentage']

X = df7[feature_cols].copy().fillna(0)

# 5. Extract and Scale features for the ML model
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 6. Apply K-Means Clustering to group booths into 4 core segments
optimal_k = 4
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
df7['Cluster_ID'] = kmeans.fit_predict(X_scaled)

# 7. Print the Raw Profile Breakdown
print("\n--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---")
profile = df7.groupby('Cluster_ID')[feature_cols].mean()
print(profile.round(2))

print("\n--- DATASET 7: BOOTH COUNT PER CLUSTER ---")
print(df7['Cluster_ID'].value_counts())

# 8. Export individual target files for campaign ground teams
for cluster_num in range(optimal_k):
    target_cols = [station_col, building_col, area_col, 'Winner_Party', 'Margin_Percentage']
    valid_target_cols = [c for c in target_cols if c in df7.columns]
    
    cluster_df = df7[df7['Cluster_ID'] == cluster_num][valid_target_cols]
    filename = f"Dataset_7_Cluster_{cluster_num}_Booths.csv"
    cluster_df.to_csv(filename, index=False)

print("\nSuccess! Campaign target files generated cleanly for all 4 clusters.")



--- DATASET 7: AVERAGE VOTE SHARE & METRICS PER CLUSTER ---
            DRAVIDAMUNNETRAKAZHAGAM_share_pct  \
Cluster_ID                                      
0                                       49.92   
1                                       33.47   
2                                       32.17   
3                                       27.20   

            ALLINDIAANNADRAVIDAMUNNETRAKAZHAGAM_share_pct  TVK_share_pct  \
Cluster_ID                                                                 
0                                                    6.55          40.44   
1                                                   10.15          51.54   
2                                                   12.20          51.35   
3                                                   11.90          56.62   

            NAAMTAMILARKATCHI_share_pct  BAHUJANSAMAJPARTY_share_pct  \
Cluster_ID                                                             
0                                  1.92    